#### Bag of Words (BoW) in NLP

Bag of Words (BoW) is a text representation technique in Natural Language Processing (NLP) that converts text into numerical vectors based on the frequency of words.

Machine learning algorithms work with numerical data. BoW converts text into numbers so that a machine learning model can process it.

<u>The basic idea is simple:</u>

1. Collect the words from a set of documents.
2. Create a vocabulary of unique words.
3. Count how many times each vocabulary word appears in each document.
4. Represent each document as a numerical vector.

BoW considers word frequency but ignores grammar and word order.

Example:

D1: <span style="color: blue;">He is a</span> good boy<br>
D2: <span style="color: blue;">She is a</span> good girl <span style="color: blue;">and a</span> good dancer<br>
D3: Boy <span style="color: blue;">and</span> girl <span style="color: blue;">are</span> good<br>

<table>
    <tr>
        <th><span style="color: blue;">Blue</span> = stop words</th>
    </tr>
</table>
<p>We can lower the case of all documents.</p>
<table>
    <tr>
        <th>Vocabulary</th>
        <th>Frequency</th>
    </tr>
    <tr>
        <td>good</td>
        <td>4</td>
    </tr>
    <tr>
        <td>boy</td>
        <td>2</td>
    </tr>
    <tr>
        <td>girl</td>
        <td>2</td>
    </tr>
    <tr>
        <td>dancer</td>
        <td>1</td>
    </tr>
</table>
<u><b>Note:</b> In case of "Binary BOW":</u>
<br>Frequency of the word in the sentence = 1 (if frequency >= 1)	
<br>Frequency of the word in the sentence = 0 (if frequency = 0)


<p><u>Vectorization (Normal Bag Of Words):</u></p>
<table>
    <tr>
        <th><span style="color: red;">Documents</span></th>
        <th>good</th>
        <th>boy</th>
        <th>girl</th>
        <th>dancer</th>
    </tr>
    <tr>
        <td><span style="color: red;">D1<span></td>
        <td>1</td>
        <td>1</td>
        <td>0</td>
        <td>0</td>
    </tr>
    <tr>
        <td><span style="color: red;">D2<span></td>
        <td>2</td>
        <td>0</td>
        <td>1</td>
        <td>1</td>
    </tr>
    <tr>
        <td><span style="color: red;">D3<span></td>
        <td>1</td>
        <td>1</td>
        <td>1</td>
        <td>0</td>
    </tr>
</table>

<p><u>Vectorization (Binary Bag Of Words):</u></p>
<table>
    <tr>
        <th><span style="color: red;">Documents</span></th>
        <th>good</th>
        <th>boy</th>
        <th>girl</th>
        <th>dancer</th>
    </tr>
    <tr>
        <td><span style="color: red;">D1<span></td>
        <td>1</td>
        <td>1</td>
        <td>0</td>
        <td>0</td>
    </tr>
    <tr>
        <td><span style="color: red;">D2<span></td>
        <td>1</td>
        <td>0</td>
        <td>1</td>
        <td>1</td>
    </tr>
    <tr>
        <td><span style="color: red;">D3<span></td>
        <td>1</td>
        <td>1</td>
        <td>1</td>
        <td>0</td>
    </tr>
</table>
<br>
All D1, D2 amd D3 have same dimension i.e. 1x4





In [28]:
from charset_normalizer import from_path

result = from_path("dataset/spamHam.csv").best()

print("Detected encoding:", result.encoding)

Detected encoding: ascii


In [29]:
import pandas as pd

messages = pd.read_csv("dataset/spamHam.csv", sep=",", 
                      names=["label", "message"], encoding="utf-8")

messages

,label,message
0,ham,"Hey, Are you free to talk this evening? No rus..."
1,spam,Alert: Your tax refund is ready. Submit your b...
2,ham,"Hey, I will message you when I get there."
3,spam,Notice: Limited-time offer! Get 70% off your n...
4,spam,Exclusive: Your mobile number has won a shoppi...
...,...,...
495,spam,Exclusive: You are eligible for a bonus reward...
496,ham,"Your parcel arrived, and I left it with the re..."
497,ham,"Hi, I forgot my umbrella at your place. Let me..."
498,ham,Quick question: We need bread and eggs for bre...


In [30]:
messages['message']

0      Hey, Are you free to talk this evening? No rus...
1      Alert: Your tax refund is ready. Submit your b...
2              Hey, I will message you when I get there.
3      Notice: Limited-time offer! Get 70% off your n...
4      Exclusive: Your mobile number has won a shoppi...
                             ...                        
495    Exclusive: You are eligible for a bonus reward...
496    Your parcel arrived, and I left it with the re...
497    Hi, I forgot my umbrella at your place. Let me...
498    Quick question: We need bread and eggs for bre...
499    Exclusive: You have won a free entry in our we...
Name: message, Length: 500, dtype: str

In [31]:
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords, wordnet
from nltk import pos_tag
import pandas as pd

messages = pd.read_csv("dataset/spamHam.csv", sep=",", 
                      names=["label", "message"], encoding="utf-8")

# Convert Penn Treebank POS tags to WordNet POS tags
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

filtered_documents=[]
for i in range(0, len(messages)) :
    # print(messages['message'][i])
    # review = re.sub('[^a-zA-Z]', ' ', messages['message'][i])
    tokens = word_tokenize(messages['message'][i])
    
    # Remove stop words before POS tagging
    filtered_tokens=[]
    for token in tokens:
        if token.isalnum() and token.lower() not in stop_words:
            filtered_tokens.append(token.lower())
    
    # POS tagging
    tagged_tokens = pos_tag(filtered_tokens)

    # Lemmatization using converted POS tags
    lemmatized_tokens = []

    for token, tag in tagged_tokens:
        wordnet_pos = get_wordnet_pos(tag)
        lemmatized_token = lemmatizer.lemmatize(token, wordnet_pos)
        lemmatized_tokens.append(lemmatized_token)
        
    # Store POS tags for lemmatized words
    filtered_documents.append(" ".join(lemmatized_tokens))

filtered_documents   

['hey free talk even rush free entire day today',
 'alert tax refund ready submit bank detail receive',
 'hey message get',
 'notice offer get 70 next purchase shop offer subject term',
 'exclusive mobile number shop voucher call redeem check detail today',
 'hey get home safely talk soon',
 'hey could check whether office open let know',
 'exclusive urgent payment require prevent service disconnect offer subject term',
 'alert account block unless confirm detail check detail today',
 'let know hi still meet lunch today rush',
 'package wait pay small fee use link release',
 'notice get instant approval personal loan apply offer subject term',
 'quick question time movie start tonight let know',
 'alert stream account hold update payment information offer subject term',
 'hey cooking dinner would like',
 'hey thanks receive photo rush',
 'let know read book recommend',
 'wallet credit bonus click withdraw',
 'hi would like tea coffee let know',
 'quick question thanks help assignment l

In [32]:
import re
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords, wordnet
from nltk import pos_tag
import pandas as pd

messages = pd.read_csv("dataset/spamHam.csv", sep=",", 
                      names=["label", "message"], encoding="utf-8")

# Convert Penn Treebank POS tags to WordNet POS tags
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

filtered_documents=[]
for i in range(0, len(messages)) :
    message = re.sub('[^a-zA-Z]', ' ', messages['message'][i])
    message = message.lower()
    tokens = message.split()
    
    # Remove stop words before POS tagging
    filtered_tokens=[]
    for token in tokens:
        if token.isalnum() and token.lower() not in stop_words:
            filtered_tokens.append(token.lower())
    
    # POS tagging
    tagged_tokens = pos_tag(filtered_tokens)

    # Lemmatization using converted POS tags
    lemmatized_tokens = []

    for token, tag in tagged_tokens:
        wordnet_pos = get_wordnet_pos(tag)
        lemmatized_token = lemmatizer.lemmatize(token, wordnet_pos)
        lemmatized_tokens.append(lemmatized_token)
        
    # Store POS tags for lemmatized words
    filtered_documents.append(" ".join(lemmatized_tokens))

filtered_documents   

['hey free talk even rush free entire day today',
 'alert tax refund ready submit bank detail receive',
 'hey message get',
 'notice limited time offer get next purchase shop offer subject term',
 'exclusive mobile number shop voucher call redeem check detail today',
 'hey get home safely talk soon',
 'hey could check whether office open let know',
 'exclusive urgent payment require prevent service disconnect offer subject term',
 'alert account block unless confirm detail check detail today',
 'let know hi still meet lunch today rush',
 'package wait pay small fee use link release',
 'notice get instant approval personal loan apply offer subject term',
 'quick question time movie start tonight let know',
 'alert stream account hold update payment information offer subject term',
 'hey cooking dinner would like',
 'hey thanks receive photo rush',
 'let know read book recommend',
 'wallet credit bonus click withdraw',
 'hi would like tea coffee let know',
 'quick question thanks help as

#### CountVectorizer (Bag Of Words)
```
class sklearn.feature_extraction.text.CountVectorizer(*, input='content', encoding='utf-8', decode_error='strict', strip_accents=None, lowercase=True, preprocessor=None, tokenizer=None, stop_words=None, token_pattern='(?u)\\b\\w\\w+\\b', ngram_range=(1, 1), analyzer='word', max_df=1.0, min_df=1, max_features=None, vocabulary=None, binary=False, dtype=<class 'numpy.int64'>)
```

Link: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html

In [39]:
# Create Bag Of Words model
from sklearn.feature_extraction.text import CountVectorizer

bagOfWords = CountVectorizer(max_features=2500, binary=True)
# bagOfWords = CountVectorizer(max_features=2500, binary=False)

X = bagOfWords.fit_transform(filtered_documents).toarray()

for arr in X:
    print(arr)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0
 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 

In [43]:
bagOfWords.vocabulary_

{'hey': np.int64(122),
 'free': np.int64(111),
 'talk': np.int64(292),
 'even': np.int64(89),
 'rush': np.int64(257),
 'entire': np.int64(85),
 'day': np.int64(62),
 'today': np.int64(304),
 'alert': np.int64(8),
 'tax': np.int64(294),
 'refund': np.int64(244),
 'ready': np.int64(238),
 'submit': np.int64(287),
 'bank': np.int64(21),
 'detail': np.int64(71),
 'receive': np.int64(239),
 'message': np.int64(170),
 'get': np.int64(113),
 'notice': np.int64(190),
 'limited': np.int64(154),
 'time': np.int64(303),
 'offer': np.int64(193),
 'next': np.int64(186),
 'purchase': np.int64(231),
 'shop': np.int64(270),
 'subject': np.int64(286),
 'term': np.int64(297),
 'exclusive': np.int64(92),
 'mobile': np.int64(176),
 'number': np.int64(192),
 'voucher': np.int64(322),
 'call': np.int64(35),
 'redeem': np.int64(242),
 'check': np.int64(42),
 'home': np.int64(126),
 'safely': np.int64(258),
 'soon': np.int64(278),
 'could': np.int64(58),
 'whether': np.int64(331),
 'office': np.int64(194),
 '

In NLP, N-grams help Bag of Words capture combinations of words instead of treating every word independently.

Why are N-grams useful for spam detection?

Consider these messages:

Message A: "You have won a prize"

Message B: "You have not won a prize"

The messages share many individual words, but their meanings differ because of the word "not".

Unigram features can identify the word "not", but they don't explicitly represent the combination "not won".
```
have won
won a
a prize
have not
not won
```

These combinations can help a machine-learning model learn patterns associated with spam and legitimate messages.

Other useful spam-related bigrams might include:
```
"claim prize"
"free entry"
"cash prize"
"call now"
"meeting today"
```

N-grams can also capture legitimate phrases, such as "see you" and "thank you".

Important: These phrases are not definitive indicators of spam. The model must learn from labeled examples, because legitimate messages can contain promotional-sounding words too.

In [46]:
# Create Bag Of Words model
from sklearn.feature_extraction.text import CountVectorizer

# bagOfWords = CountVectorizer(max_features=2500, binary=True, ngram_range=(1,1))
bagOfWords = CountVectorizer(max_features=2500, binary=True, ngram_range=(1,2))
# bagOfWords = CountVectorizer(max_features=2500, binary=True, ngram_range=(1,3))
# bagOfWords = CountVectorizer(max_features=2500, binary=True, ngram_range=(2,3))
# bagOfWords = CountVectorizer(max_features=2500, binary=False)

X = bagOfWords.fit_transform(filtered_documents).toarray()

for arr in X:
    print(arr)

[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ...

In [47]:
bagOfWords.vocabulary_

{'hey': np.int64(489),
 'free': np.int64(440),
 'talk': np.int64(1171),
 'even': np.int64(349),
 'rush': np.int64(1076),
 'entire': np.int64(339),
 'day': np.int64(272),
 'today': np.int64(1203),
 'hey free': np.int64(500),
 'free talk': np.int64(447),
 'talk even': np.int64(1172),
 'even rush': np.int64(351),
 'rush free': np.int64(1077),
 'free entire': np.int64(442),
 'entire day': np.int64(340),
 'day today': np.int64(276),
 'alert': np.int64(27),
 'tax': np.int64(1180),
 'refund': np.int64(1040),
 'ready': np.int64(1018),
 'submit': np.int64(1159),
 'bank': np.int64(108),
 'detail': np.int64(293),
 'receive': np.int64(1020),
 'alert tax': np.int64(59),
 'tax refund': np.int64(1181),
 'refund ready': np.int64(1042),
 'ready submit': np.int64(1019),
 'submit bank': np.int64(1160),
 'bank detail': np.int64(110),
 'detail receive': np.int64(297),
 'message': np.int64(747),
 'get': np.int64(454),
 'hey message': np.int64(509),
 'message get': np.int64(748),
 'notice': np.int64(800),
 '

<u>Advantages of Bag Of Words:</u>
1. All vectors are of equal size i.e., vocabulary size (Important for ML Algorithm)
2. Simple and intutive.

<u>Disadvantages of Bag Of Words:</u>
1. Sparse vectors (when vocabulary count is high)
2. Ordering/sequence of the words in the sentence might change because of their frequencies. This may change the meaning of a sentence.
3. If any new word comes (other than the vocabulary used during training) then the word is ignored during vectorization.
4. Symantics is not getting captured. 
    What is the most important word in the sentence ? 
    How close sentences are ?
    <br> Eg., D1: The food is good.
    <br>      D2: The food is not good.
    <br> Both D1 and D2 when they will be plotted in 2D  plane, they will be very close and angle between them will be very less. But in reality angle between them should be 180 degree as they are opposite in terms of meaning.

In [52]:
from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "good boy",
    "good girl good dancer",
    "boy girl good"
]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)

print(vectorizer.get_feature_names_out())
print(X.toarray())

['boy' 'dancer' 'girl' 'good']
[[1 0 0 1]
 [0 1 1 2]
 [1 0 1 1]]


In the above example we can see that vocabulary order is alphabatical. It is not based on frequency.

<b>Note:</b> CountVectorizer sorts the vocabulary in alphabatical order.